# LeIsaac SO-101 PickOrange VLA Play

在 Isaac Sim 里跑 SO-101 单臂从厨房桌面抓橙子放盘子，对比多条 VLA 推理链路。
_Run SO-101 picking oranges onto a plate in Isaac Sim, comparing multiple VLA inference pipelines side-by-side._

| 子章节 | 模型 | server | port |
| --- | --- | --- | --- |
| §1 | `hi-space/GR00T-N1.6-3B-Pick-Orange` | GR00T N1.6 (`run_gr00t_server.py`) | 5555 |
| §2 | `LightwheelAI/leisaac-pick-orange-v0` | GR00T N1.5 (`inference_service.py`) | 5555 |
| §3.3 | **`wsagi/ACT-PickOrange` (ours)** + `shadowHokage/act_policy` | LeRobot async (`policy_server`) | 8080 |
| §3.4 | **`wsagi/DiffusionPolicy-PickOrange` (ours)** — DDIM 32-step swap | LeRobot async | 8080 |
| §3.5 | **`wsagi/SmolVLA-PickOrange` (ours)** + `edge-inference/smolvla-so101-pick-orange` | LeRobot async | 8080 |
| §4 | `hi-space/GR00T-N1.7-3B-Pick-Orange` | GR00T N1.7 | 5555 |

📊 **横评结果表 / Benchmark results** 见 [`README.md`](README.md#benchmark--7-baselines--3-rounds--3-oranges) (简要) 或 [`LeIsaac/README.md`](LeIsaac/README.md#2-pickorange-多策略横评) (含 per-round detail + GPU util + VRAM)。

**前提条件 / Prerequisites**：
1. 在 `isaaclab-experience/` 目录下启动 jupyter
2. 启动前已激活 conda 环境：`conda activate isaaclab`
3. 已 `bash scripts/apply_leisaac_patches.sh`（如果首次拉 submodule）
4. 每个推理 cell 需要 Isaac Sim 弹窗，确保 `DISPLAY` 可用

**端口冲突注意 / Port note**：§1/§2/§4 共用 :5555（GR00T 系列），任一时刻只能起一个。§3 的 LeRobot server (:8080) 与 GR00T 互不干扰，可共存。

**⚠️ ACT/DP 重要 inference 配置 / Critical inference setting for ACT/DP**：ACT (chunk_size=100) 必须 `policy_action_horizon=32`（默认 16 卡第 2 颗）；DP 必须 hot-swap ckpt 到 DDIM 32-step（DDPM 100-step 太慢）。详见各子章节。
_ACT (chunk_size=100) requires `policy_action_horizon=32` (the default 16 deadlocks on the 2nd orange); DP must hot-swap ckpt to DDIM 32-step (DDPM 100-step is too slow). Details in each sub-section._


## 0) 预检查

确保 LeIsaac submodule 文件齐全。

In [ ]:
!test -f scripts/evaluation/policy_inference.py && test -f assets/robots/so101_follower.usd && test -f assets/scenes/kitchen_with_orange/scene.usd && python scripts/evaluation/policy_inference.py --help | head -n 20

## 1) GR00T N1.6 fine-tune (hi-space)

**模型**：`hi-space/GR00T-N1.6-3B-Pick-Orange` —— 社区在 LeIsaac PickOrange 数据上对 `nvidia/GR00T-N1.6-3B` 进行 fine-tune。N1.6 用 flow-matching action head，相比 N1.5 的 DiT diffusion 更新一代。

⚠ 与 §1.1 N1.5 共用 ZMQ :5555，先确保 N1.5 server 已停（`!bash scripts/policy_server.sh stop gr00t-n15`）。

### 1.1) 一键下载 fine-tuned ckpt（HF cache 已有则跳过）

In [ ]:
!bash ../scripts/download_hf_model.sh hi-space/GR00T-N1.6-3B-Pick-Orange

### 1.2) 一键启动 GR00T N1.6 推理服务（ZMQ :5555）

走 `server/start_server.sh --gr00t-only`，自动注入 `GR00T_MODEL_PATH` 和 `GR00T_EMBODIMENT_TAG=NEW_EMBODIMENT`。

In [ ]:
!bash scripts/policy_server.sh start gr00t-n16

### 1.3) 运行 SO-101 PickOrange 仿真实时推理

`--policy_type=gr00tn1.6` 走 `Gr00t16ServicePolicyClient`。

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/evaluation/policy_inference.py --task=LeIsaac-SO101-PickOrange-v0 --eval_rounds=1 --episode_length_s=120 --policy_type=gr00tn1.6 --policy_host=127.0.0.1 --policy_port=5555 --policy_timeout_ms=15000 --policy_language_instruction='Pick the orange to the plate' --policy_action_horizon=16 --device=cuda --enable_cameras

### 1.4) 停止 GR00T N1.6 推理服务

In [ ]:
!bash scripts/policy_server.sh stop gr00t-n16

## 2) GR00T N1.5 + LightwheelAI fine-tune

NVIDIA Isaac-GR00T N1.5（3B 参数，DiT diffusion action head）+ LightwheelAI 在仿真录制数据上 fine-tune 的 `leisaac-pick-orange-v0` checkpoint。

实测：1/1 成功率，机器人完成 pick-and-place 完整动作。

### 2.1) 一键下载 fine-tuned ckpt（~7.4 GB，已有则跳过）

In [ ]:
!bash ../scripts/download_hf_model.sh LightwheelAI/leisaac-pick-orange-v0


### 2.2) 一键启动 GR00T N1.5 推理服务（ZMQ :5555）

冷启动需加载 Eagle backbone + DiT head，约 20–30s；已启动则 idempotent skip。


In [ ]:
!bash scripts/policy_server.sh start gr00t-n15


### 2.3) 运行 SO-101 PickOrange 仿真实时推理

Isaac Sim 会弹窗显示 SO-101 单臂；客户端连 `:5555` 调 GR00T N1.5 推理。`-u` 是为了让 Episode/success 日志实时刷出来（Isaac Sim 退出会跳过 stdout flush）。


In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/evaluation/policy_inference.py --task=LeIsaac-SO101-PickOrange-v0 --eval_rounds=1 --episode_length_s=120 --policy_type=gr00tn1.5 --policy_host=127.0.0.1 --policy_port=5555 --policy_timeout_ms=15000 --policy_language_instruction='Pick up the orange and place it on the plate' --policy_action_horizon=16 --device=cuda --enable_cameras


### 2.4) 停止 GR00T N1.5 推理服务（释放显存）

In [ ]:
!bash scripts/policy_server.sh stop gr00t-n15


## 3) LeRobot async-inference (ACT / Diffusion Policy / SmolVLA / 任意 LeRobot policy)

LeRobot v0.4+ 的 `policy_server` 是**通用 server**：client 通过 `RemotePolicyConfig` 告诉它该加载哪个 ckpt，所以一个 :8080 server 进程能同时服务 ACT、Diffusion Policy、SmolVLA、pi0… 任何 LeRobot 框架的 policy。
_LeRobot v0.4+'s `policy_server` is a **generic server**: the client tells it which ckpt to load via `RemotePolicyConfig`, so one :8080 process can serve ACT / Diffusion Policy / SmolVLA / pi0 / any LeRobot policy._

区分发生在 client 端：`--policy_type=lerobot-<model_type>` + `--policy_checkpoint_path=<repo_id>`。下面演示三类 client：
_Differentiation happens on the client side: `--policy_type=lerobot-<model_type>` + `--policy_checkpoint_path=<repo_id>`. Three client examples below:_

| Client | model_type | ckpt（自训 / ours 标 ⭐）| 输入 |
| --- | --- | --- | --- |
| ACT | `lerobot-act` | ⭐ `wsagi/ACT-PickOrange` / `shadowHokage/act_policy` | 纯 vision + state → action chunk，~80M |
| Diffusion Policy | `lerobot-diffusion` | ⭐ `wsagi/DiffusionPolicy-PickOrange` (DDIM 32) | vision + state → 8-step action chunk，~267M |
| SmolVLA | `lerobot-smolvla` | `edge-inference/smolvla-so101-pick-orange` | VLM（语言+视觉） + Action Expert，~450M |

### 3.1) 安装 LeIsaac LeRobot client 依赖（首次执行后可跳过）

In [ ]:
!pip install -e "source/leisaac[lerobot-async]"

### 3.2) 一键启动 LeRobot 推理服务（端口 :8080）

Idempotent：已起则跳过。SmolVLA 和 ACT 都连这一个 server。

In [ ]:
!bash scripts/policy_server.sh start lerobot

In [ ]:
import os

# Per-model action horizon — passed via env var to `!` shell cells below.
# 每个模型 chunk query 后执行多少 action 再重新 query — 通过 env var 传给下面 !cell。
#
# 防呆 / Safety: 下面 inference cell 用 ${HORIZON_XXX:-DEFAULT} bash 语法，
# 即便没跑本 cell 也能用文档化默认值，不会因 env var 未设报 invalid int 错。
# The inference cells use `${HORIZON_XXX:-DEFAULT}` bash syntax — even if this
# cell never ran, the documented defaults kick in. No NameError / empty-string crashes.
#
# 经验法则 / Rule of thumb: horizon ≈ chunk_size × 0.3 ~ 0.7
#   - 太低 / Too low  → 走不到 macro-motion 段，卡 approach-loop / can't reach macro-motion, stuck in approach loop
#   - 太高 / Too high → 开环执行，误差累积撞东西或散架 / open-loop drift, errors accumulate
#
# 实测 sweet spot / Measured sweet spots (this codebase):
#   - ACT             (chunk_size=100):  32  ✅ 1/1 success. Range tested: 8 ❌ / 16 🟡(卡 2nd) / 32 ✅
#   - DiffusionPolicy (n_action_steps=8): 16  (client 自动截到 8 / auto-capped to chunk=8 by server)
#   - SmolVLA         (chunk_size=50):   16  (未做系统 sweep / not systematically swept)
#   - π0.5            (chunk_size=50):   35  ✅ 1 颗落盘. Range tested: 16 ❌(卡 grasp) / 25 🟡(grasp 后松) / 35 ✅
#     —— π0.5 不在本 notebook，跑法见 server/eval_pi05.sh / runs from server/eval_pi05.sh, not this notebook
#
# 改完工作流 / Workflow:
#   (1) 改下面 os.environ 值
#   (2) 重跑本 cell 让 env var 写入 kernel
#   (3) 重跑对应 §3.3 / §3.4 / §3.5 的 inference cell
#   不重跑本 cell 也 OK，inference cell 会用 ${VAR:-DEFAULT} 里的默认值 / Skipping this cell is fine — defaults kick in.

os.environ["HORIZON_ACT"]     = "32"   # chunk_size=100. Default LeIsaac.ipynb 老值 16 会卡第 2 颗橙子 — 必须升到 32
os.environ["HORIZON_DP"]      = "16"   # client horizon ≥ 8 都等效（model n_action_steps=8）
os.environ["HORIZON_SMOLVLA"] = "50"   # = chunk_size, full receding window (实测 wsagi/SmolVLA 1/3 ✅, 5/9 🍊)

print(f"HORIZON_ACT     = {os.environ['HORIZON_ACT']}")
print(f"HORIZON_DP      = {os.environ['HORIZON_DP']}")
print(f"HORIZON_SMOLVLA = {os.environ['HORIZON_SMOLVLA']}")

### 3.3) ACT policy client — `wsagi/ACT-PickOrange` (ours) + `shadowHokage/act_policy`

Action Chunking Transformer (~80M)，LeRobot 框架下的轻量 imitation learning baseline（纯 vision + state → action chunk）。两个 ckpt 可选：
_ACT (~80M), light-weight imitation learning baseline in LeRobot (vision + state → action chunk). Two ckpts available:_

- [`wsagi/ACT-PickOrange`](https://huggingface.co/wsagi/ACT-PickOrange) — 我们自训的 10k step ckpt（复刻 shadowHokage 配方），验证 **1/1 success @ horizon=32** / Our 10k-step reproduction of the shadowHokage recipe, validated 1/1 success.
- [`shadowHokage/act_policy`](https://huggingface.co/shadowHokage/act_policy) — 社区参考 ckpt / community reference.

两个 ckpt 的 `input_features` 都直接用 `observation.images.front` / `observation.images.wrist`，与 LeIsaac sim 摄像头键名一致，无需 rename。

⚠️ **关键 inference 配置 / Critical inference setting**：`--policy_action_horizon=32`（**不是**老默认 16）+ `--step_hz=30`（对齐 dataset 30Hz）。默认 horizon=16 会让模型卡在第 2 颗橙子（爪子抖 / muting effect）。完整诊断见 [`LeIsaac/docs/training/act_eval_debug_postmortem.html`](LeIsaac/docs/training/act_eval_debug_postmortem.html)。
_⚠️ Use `--policy_action_horizon=32` (NOT the legacy 16) and `--step_hz=30` (matches dataset). horizon=16 deadlocks on the 2nd orange._

**3.3a) 项目自训 ACT / Our ACT — `wsagi/ACT-PickOrange`**

In [ ]:
!bash ../scripts/download_hf_model.sh wsagi/ACT-PickOrange

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/evaluation/policy_inference.py --task=LeIsaac-SO101-PickOrange-v0 --eval_rounds=1 --episode_length_s=120 --step_hz=30 --policy_type=lerobot-act --policy_host=127.0.0.1 --policy_port=8080 --policy_timeout_ms=15000 --policy_language_instruction='Pick up the orange and place it on the plate' --policy_checkpoint_path=wsagi/ACT-PickOrange --policy_action_horizon=${HORIZON_ACT:-32} --device=cuda --enable_cameras

**3.3b) 社区参考 ACT / Community reference — `shadowHokage/act_policy`**

In [ ]:
!bash ../scripts/download_hf_model.sh shadowHokage/act_policy

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/evaluation/policy_inference.py --task=LeIsaac-SO101-PickOrange-v0 --eval_rounds=1 --episode_length_s=120 --step_hz=30 --policy_type=lerobot-act --policy_host=127.0.0.1 --policy_port=8080 --policy_timeout_ms=15000 --policy_language_instruction='Pick the orange to the plate' --policy_checkpoint_path=shadowHokage/act_policy --policy_action_horizon=${HORIZON_ACT:-32} --device=cuda --enable_cameras

### 3.4) Diffusion Policy client — `wsagi/DiffusionPolicy-PickOrange` (ours)

[`wsagi/DiffusionPolicy-PickOrange`](https://huggingface.co/wsagi/DiffusionPolicy-PickOrange) —— 项目自训 LeRobot Diffusion Policy（~267M，UNet 1D + ResNet18 vision），**已 hot-swap 到 DDIM 32-step inference**（不重训，直接改 ckpt `config.json`：`noise_scheduler_type: DDIM` + `num_inference_steps: 32`）。
_Our Diffusion Policy (~267M, UNet 1D + ResNet18 vision), **with DDIM 32-step inference hot-swapped into the ckpt config** (no retraining: edit `noise_scheduler_type: DDIM` + `num_inference_steps: 32` in `config.json`)._

**实测推理 / Inference latency**：DDPM 100-step 393 ms/chunk → DDIM 32-step **147 ms/chunk**，slowdown 2.96x → 1.1x，RTX 4090 实时跑得动。详细 postmortem 见 [`LeIsaac/docs/training/dp_inference_speedup_and_dynamic_timeout.html`](LeIsaac/docs/training/dp_inference_speedup_and_dynamic_timeout.html)。

**结果 / Result**：⚠️ 1-2/3 — DDIM swap 解锁实时性，但 dataset OOD on 3rd orange 仍是上限 / DDIM swap unlocks realtime but dataset OOD on the 3rd orange remains the ceiling.

In [ ]:
!bash ../scripts/download_hf_model.sh wsagi/DiffusionPolicy-PickOrange

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/evaluation/policy_inference.py --task=LeIsaac-SO101-PickOrange-v0 --eval_rounds=1 --episode_length_s=120 --step_hz=60 --policy_type=lerobot-diffusion --policy_host=127.0.0.1 --policy_port=8080 --policy_timeout_ms=10000 --policy_language_instruction='Pick up the orange and place it on the plate' --policy_checkpoint_path=wsagi/DiffusionPolicy-PickOrange --policy_action_horizon=${HORIZON_DP:-16} --device=cuda --enable_cameras

### 3.5) SmolVLA fine-tune client — `wsagi/SmolVLA-PickOrange` (ours) + `edge-inference/smolvla-so101-pick-orange`

SmolVLA (~450M)，SmolVLM2-500M-Video-Instruct backbone + Action Expert，LeRobot 框架下的小型 VLM-based policy（vision + language + state → action chunk）。两个 ckpt 可选：
_SmolVLA (~450M), SmolVLM2-500M-Video-Instruct backbone + Action Expert in LeRobot. Two ckpts available:_

- [`wsagi/SmolVLA-PickOrange`](https://huggingface.co/wsagi/SmolVLA-PickOrange) — 我们自训的 30k step ckpt（full-param fine-tune，无 LoRA），实测 **1/3 rounds, 5/9 🍊** / Our 30k-step full-param fine-tune.
- [`edge-inference/smolvla-so101-pick-orange`](https://huggingface.co/edge-inference/smolvla-so101-pick-orange) — 社区参考 ckpt，实测 **0/3 rounds, 0/9 🍊** / Community reference (weaker).

两个 ckpt 的 `input_features` 都用自然 keys (`observation.images.front` / `.wrist`)，与 LeIsaac sim 一致，无需 rename。
_Both ckpts use natural feature keys matching the sim — no rename needed._

⚠️ **关键 inference 配置 / Critical inference setting**：`--policy_action_horizon=50`（= chunk_size，full receding window）+ `--step_hz=30`（对齐 dataset 30Hz）。


**3.5a) 项目自训 SmolVLA / Our SmolVLA — `wsagi/SmolVLA-PickOrange`**

In [ ]:
!bash ../scripts/download_hf_model.sh wsagi/SmolVLA-PickOrange

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/evaluation/policy_inference.py --task=LeIsaac-SO101-PickOrange-v0 --eval_rounds=1 --episode_length_s=120 --step_hz=30 --policy_type=lerobot-smolvla --policy_host=127.0.0.1 --policy_port=8080 --policy_timeout_ms=15000 --policy_language_instruction='Pick up the orange and place it on the plate' --policy_checkpoint_path=wsagi/SmolVLA-PickOrange --policy_action_horizon=${HORIZON_SMOLVLA:-50} --device=cuda --enable_cameras

**3.5b) 社区参考 SmolVLA / Community reference — `edge-inference/smolvla-so101-pick-orange`**

In [ ]:
!bash ../scripts/download_hf_model.sh edge-inference/smolvla-so101-pick-orange

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/evaluation/policy_inference.py --task=LeIsaac-SO101-PickOrange-v0 --eval_rounds=1 --episode_length_s=120 --policy_type=lerobot-smolvla --policy_host=127.0.0.1 --policy_port=8080 --policy_timeout_ms=15000 --policy_language_instruction='Pick the orange to the plate' --policy_checkpoint_path=edge-inference/smolvla-so101-pick-orange --policy_action_horizon=${HORIZON_SMOLVLA:-50} --device=cuda --enable_cameras

### 3.6) 停止 LeRobot 推理服务（释放显存）

不影响 GR00T `:5555`。

In [ ]:
!bash scripts/policy_server.sh stop lerobot

## 4) GR00T N1.7 fine-tune (hi-space) ⛔ infra 待搭

**模型**：`hi-space/GR00T-N1.7-3B-Pick-Orange` —— 上游 NVIDIA `Isaac-GR00T@n1.7-release`（commit `23ace64`）发布 N1.7 后，社区在 LeIsaac PickOrange 数据上的 fine-tune。

**当前状态**：ckpt 已下载到 HF cache（24GB，含 optimizer state，纯推理只需 12.6GB 的 3 个 safetensors）。但**推理端没法直接跑**，缺三件套：

1. **Isaac-GR00T-N1.7 仓库**：本地 `~/work/Isaac-GR00T` 还在 N1.6 commit `7d5a455`，不识别 `model_type: Gr00tN1d7`。需要：
   ```bash
   cd ~/work && git clone https://github.com/NVIDIA/Isaac-GR00T Isaac-GR00T-N1.7
   cd ~/work/Isaac-GR00T-N1.7 && git checkout n1.7-release
   ```
2. **新 conda env `gr00t-n17`**：torch / transformers / flash-attn 版本组合大概率和 N1.5/N1.6 不同。参考 `Isaac-GR00T-N1.7/pyproject.toml` 装一遍。
3. **LeIsaac client 端补丁**：
   - `source/leisaac/leisaac/policy/service_policy_clients.py` 加 `Gr00t17ServicePolicyClient`（参照 `Gr00t16ServicePolicyClient` 改）
   - `scripts/evaluation/policy_inference.py` 加 `gr00tn1.7` 分支
   - `scripts/policy_server.sh` 加 `gr00t-n17` backend，调用新仓库的 `gr00t/eval/run_gr00t_server.py`

上面三步搞完，下面 4.1~4.4 cell 即可填上对应命令。现在保留为占位骨架。

### 4.1) 一键下载 fine-tuned ckpt（HF cache 已有则跳过）

In [ ]:
!bash ../scripts/download_hf_model.sh hi-space/GR00T-N1.7-3B-Pick-Orange

### 4.2) 一键启动 GR00T N1.7 推理服务（ZMQ :5555） ⛔ 暂未实现

```bash
# 计划：
# !bash scripts/policy_server.sh start gr00t-n17
```

### 4.3) 运行 SO-101 PickOrange 仿真实时推理 ⛔ 暂未实现

```bash
# 计划：
# !PYTHONUNBUFFERED=1 python -u scripts/evaluation/policy_inference.py \
#     --task=LeIsaac-SO101-PickOrange-v0 --eval_rounds=1 --episode_length_s=120 \
#     --policy_type=gr00tn1.7 --policy_host=127.0.0.1 --policy_port=5555 \
#     --policy_action_horizon=16 --device=cuda --enable_cameras
```

---

## 🧹 一键清理 / Emergency cleanup

如果 Isaac Sim 窗口卡死、`policy_inference` grandchild 没被 `timeout` 杀干净、显存被吃光，跑下面 cell 一键收拾：
_If an Isaac Sim window hangs, a `policy_inference` grandchild survives `timeout`, or GPU memory is exhausted — run the cell below to kill everything and verify GPU is free._

- 杀全部 `policy_inference` + `isaac-sim` + `kit.py` 进程（含 grandchild）
- 停 LeRobot `:8080` / GR00T `:5555` / π0.5 `:5556` 三类 server
- 报告剩余进程 + 占用端口 + GPU 显存

_Kills all `policy_inference` / `isaac-sim` / `kit.py` processes (including grandchildren), stops LeRobot/GR00T/π0.5 servers, then reports residual processes, listening ports, and GPU memory._

In [ ]:
!bash scripts/cleanup_eval.sh

### 4.4) 停止 GR00T N1.7 推理服务 ⛔ 暂未实现